Checklist

 - Add GNN layer (currently 2, need 3)
 - Reduce dimension of GNN layer
 - Add relevant features in featurizer
 - Check dimensions at each layer
 - Hyperparameter tuning

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem.Draw import MolsToGridImage

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")

np.random.seed(42)

In [1]:
import torch
from torch.nn import Linear
import torch.nn.functional as F 
from torch_geometric.nn import GCNConv, TopKPooling
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp
embedding_size = 64

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

c:\Users\spran\anaconda3\envs\drug\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [3]:
import numpy as np
from rdkit import Chem
from rdkit.Chem.rdchem import BondType

class Featurizer:
    def __init__(self, allowable_sets, direct_features=None):
        """
        Base featurizer class. 
        - `allowable_sets`: Dict of categorical features (one-hot encoded).
        - `direct_features`: Dict of numerical features (used as direct values).
        """
        self.dim = 0
        self.features_mapping = {}
        self.direct_features = direct_features if direct_features else {}

        # One-hot encoded categorical features
        for k, s in allowable_sets.items():
            s = sorted(list(s))
            self.features_mapping[k] = dict(zip(s, range(self.dim, len(s) + self.dim)))
            self.dim += len(s)

        # Direct numerical features (not one-hot)
        for k in self.direct_features.keys():
            self.features_mapping[k] = self.dim
            self.dim += 1

    def encode(self, inputs):
        output = np.zeros((self.dim,))

        # One-hot encoded features
        for name_feature, feature_mapping in self.features_mapping.items():
            if name_feature in self.direct_features:  # Skip direct features in one-hot encoding
                continue
            feature = getattr(self, name_feature)(inputs)
            if feature not in feature_mapping:
                continue
            output[feature_mapping[feature]] = 1.0

        # Direct numerical features
        for name_feature, index in self.direct_features.items():
            feature = getattr(self, name_feature)(inputs)
            output[index] = feature  # Assign directly

        return output


class AtomFeaturizer(Featurizer):
    def __init__(self, allowable_sets, direct_features):
        super().__init__(allowable_sets, direct_features)

    def symbol(self, atom):
        return atom.GetSymbol()

    def n_hydrogens(self, atom):
        return atom.GetTotalNumHs() if atom.HasProp("_TotalNumHs") else 0

    def hybridization(self, atom):
        return atom.GetHybridization().name.lower()

    def is_aromatic(self, atom):
        return atom.GetIsAromatic()

    def n_valence(self, atom):
        return atom.GetTotalValence()

    def formal_charge(self, atom):
        return atom.GetFormalCharge()  # Direct feature, not one-hot

    def atomic_number(self, atom):
        return atom.GetAtomicNum()  # Direct feature


class BondFeaturizer(Featurizer):
    def __init__(self, allowable_sets, direct_features):
        super().__init__(allowable_sets, direct_features)

    def bond_type(self, bond):
        return bond.GetBondType().name.lower()

    def conjugated(self, bond):
        return bond.GetIsConjugated()

    def bond_order(self, bond):
        """Returns a numeric bond order instead of a categorical one-hot encoding."""
        bond_orders = {
            BondType.SINGLE: 1.0,
            BondType.DOUBLE: 2.0,
            BondType.TRIPLE: 3.0,
            BondType.AROMATIC: 1.5,
        }
        return bond_orders.get(bond.GetBondType(), 0.0)


# Instantiate with updated feature sets
atom_featurizer = AtomFeaturizer(
    allowable_sets={
        "symbol": {"B", "Br", "C", "Ca", "Cl", "F","Ga", "H", "I", "N", "Na", "O", "P", "S","Sb","Se", "Mo", "Nb"},
        "n_hydrogens": {0, 1, 2, 3, 4},
        "hybridization": {"s", "sp", "sp2", "sp3"},
    },
    direct_features={
        "formal_charge": None,  # This will be assigned a direct index in `Featurizer`
        "atomic_number": None,
        # "n_valence": None,  # Direct numeric feature
        "is_aromatic": None,  # Direct boolean feature
        "n_hydrogens": None,  # Direct numeric feature
    }
)

bond_featurizer = BondFeaturizer(
    allowable_sets={
        "bond_type": {"single", "double", "triple", "aromatic"},
        "conjugated": {True, False},
    },
    direct_features={
        "bond_order": None,  # Direct numeric feature
    }
)

# Get the new feature dimensions
node_dim = 21 #atom_featurizer.dim
edge_dim = bond_featurizer.dim

print(f"Node feature dimension: {node_dim}")
print(f"Edge feature dimension: {edge_dim}")


Node feature dimension: 21
Edge feature dimension: 7


In [4]:
ATOM_LIST = ["H", "B", "C", "N", "O", "F", "Na", "P", "S", "Cl", 
             "Ca", "I", "Br", "Se", "Mo", "Nb", "Ga", "Sb"]

ATOMIC_MASS = {
    "H": 1.008, "B": 10.81, "C": 12.01, "N": 14.01, "O": 16.00,
    "F": 18.998, "Na": 22.99, "P": 30.97, "S": 32.06, "Cl": 35.45,
    "Ca": 40.08, "I": 126.90, "Br": 79.90, "Se": 78.96, "Mo": 95.95,
    "Nb": 92.91, "Ga": 69.72, "Sb": 121.76
}

VALENCY = {
    "H": 1, "B": 3, "C": 4, "N": 3, "O": 2, "F": 1,
    "Na": 1, "P": 3, "S": 2, "Cl": 1, "Ca": 2,
    "I": 1, "Br": 1, "Se": 2, "Mo": 6, "Nb": 5,
    "Ga": 3, "Sb": 5
}

ELECTRONEGATIVITY = {
    "H": 2.20, "B": 2.04, "C": 2.55, "N": 3.04, "O": 3.44, "F": 3.98,
    "Na": 0.93, "P": 2.19, "S": 2.58, "Cl": 3.16, "Ca": 1.00,
    "I": 2.66, "Br": 2.96, "Se": 2.55, "Mo": 2.16, "Nb": 1.6,
    "Ga": 1.81, "Sb": 2.05
}

class BasicAtomFeaturizer:
    def __init__(self, atom_list=ATOM_LIST):
        self.atom_list = atom_list
        self.atom_index = {a: i for i, a in enumerate(atom_list)}

    def encode(self, atom):
        symbol = atom.GetSymbol()
        
        # One-hot symbol
        one_hot = np.zeros(len(self.atom_list))
        if symbol in self.atom_index:
            one_hot[self.atom_index[symbol]] = 1.0
        
        # Lookups with defaults
        mass = ATOMIC_MASS.get(symbol, 0.0)
        valency = VALENCY.get(symbol, 0.0)
        eneg = ELECTRONEGATIVITY.get(symbol, 0.0)

        return np.concatenate([one_hot, [mass, valency, eneg]])
    
basic_featurizer = BasicAtomFeaturizer()


In [5]:
import torch
import numpy as np
from rdkit import Chem
from torch_geometric.data import Data

# Ensure you define atom_featurizer and bond_featurizer before using them

def molecule_from_smiles(smiles):
    """Convert SMILES to RDKit molecule with error handling."""
    molecule = Chem.MolFromSmiles(smiles, sanitize=False)
    flag = Chem.SanitizeMol(molecule, catchErrors=True)
    if flag != Chem.SanitizeFlags.SANITIZE_NONE:
        Chem.SanitizeMol(molecule, sanitizeOps=Chem.SanitizeFlags.SANITIZE_ALL ^ flag)
    Chem.AssignStereochemistry(molecule, cleanIt=True, force=True)
    return molecule

def graph_from_molecule(molecule):
    """Convert RDKit molecule to PyTorch Geometric graph representation."""
    atom_features = []
    bond_features = []
    edge_index = []
    edge_attr = []

    # Chem.SanitizeMol(molecule)  # Ensure the molecule is sanitized

    for atom in molecule.GetAtoms():
        atom_features.append(basic_featurizer.encode(atom))  # Encode atom features

    for bond in molecule.GetBonds():
        start, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.append([start, end])
        edge_index.append([end, start])  # Ensure undirected edges
        bond_features.append(bond_featurizer.encode(bond))  # Bond features
        bond_features.append(bond_featurizer.encode(bond))  # Reverse edge

    # Convert to PyTorch tensors
    x = torch.tensor(atom_features, dtype=torch.float)
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(bond_features, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

def graphs_from_smiles(smiles_list):
    """Convert a list of SMILES strings to PyTorch Geometric Data objects."""
    graphs = []
    for smiles in smiles_list:
        molecule = molecule_from_smiles(smiles)
        graph = graph_from_molecule(molecule)
        graphs.append(graph)
    return graphs  # This can be used with a PyG DataLoader


In [2]:
from pymatgen.core import Structure
from pymatgen.io.cif import CifParser
from pymatgen.analysis.local_env import CutOffDictNN
from pymatgen.core.periodic_table import Element
from scipy.spatial import distance_matrix
from rdkit import Chem
from rdkit.Chem import AllChem, Draw

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

def load_molecule(base_path, cutoff=3.0, max_atoms=200):
    """
    Try to load a molecule/structure from base_path.cif or base_path.sdf
    Returns an RDKit Mol or None.
    """
    cif_path = base_path + ".cif"
    sdf_path = base_path + ".sdf"

    mol = None

    if os.path.exists(cif_path):
        try:
            mol = cif_to_mol(cif_path, cutoff=cutoff, max_atoms=max_atoms)
            print(f"Loaded CIF: {cif_path} with {mol.GetNumAtoms()} atoms")
        except Exception as e:
            print(f"❌ Failed to parse CIF {cif_path}: {e}")

    elif os.path.exists(sdf_path):
        try:
            suppl = Chem.SDMolSupplier(sdf_path, removeHs=False)
            mols = [m for m in suppl if m is not None]
            if mols:
                mol = mols[0]
                print(f"Loaded SDF: {sdf_path} with {mol.GetNumAtoms()} atoms")
            else:
                print(f"⚠️ No valid molecules in {sdf_path}")
        except Exception as e:
            print(f"❌ Failed to parse SDF {sdf_path}: {e}")

    else:
        print(f"⚠️ No CIF or SDF found for {base_path}")

    return mol

def cif_to_mol(cif_file, cutoff=3.0, max_atoms=200):
    """
    Convert CIF to an RDKit molecule (approximate).
    cutoff : max distance in Å to consider bonds
    max_atoms : safeguard to avoid gigantic cells
    """
    structure = Structure.from_file(cif_file)

    mol = Chem.RWMol()

    # --- add atoms ---
    for site in structure:
        element = site.specie.symbol
        atom = Chem.Atom(element)
        mol.AddAtom(atom)

    if mol.GetNumAtoms() > max_atoms:
        print(f"⚠️ CIF {cif_file} has {mol.GetNumAtoms()} atoms, truncating to first {max_atoms}.")
        # Drop atoms beyond limit
        while mol.GetNumAtoms() > max_atoms:
            mol.RemoveAtom(mol.GetNumAtoms()-1)

    # --- add bonds (distance cutoff) ---
    n_atoms = mol.GetNumAtoms()
    for i in range(n_atoms):
        for j in range(i + 1, n_atoms):
            dist = structure.get_distance(i, j)
            if dist < cutoff:
                if mol.GetBondBetweenAtoms(i, j) is None:
                    mol.AddBond(i, j, Chem.BondType.SINGLE)

    # --- coordinates ---
    conf = Chem.Conformer(mol.GetNumAtoms())
    for idx, site in enumerate(structure[:n_atoms]):
        x, y, z = site.coords
        conf.SetAtomPosition(idx, (float(x), float(y), float(z)))
    mol.AddConformer(conf)

    return mol

In [5]:
path = "./CIF_files/graphene"
drug = "./SDF_files/fluorouracil"

mol = load_molecule(path, cutoff=2.0, max_atoms=200)
drug = load_molecule(drug, cutoff=2.0, max_atoms=200)

from rdkit import Chem

def safe_molecule_features(mol):
    feats = []
    conf = None
    try:
        conf = mol.GetConformer()
    except Exception:
        pass  # no coords available

    for atom in mol.GetAtoms():
        idx = atom.GetIdx()
        atom_feats = {}

        # Symbol
        try:
            atom_feats["symbol"] = atom.GetSymbol()
        except Exception:
            atom_feats["symbol"] = None

        # Atomic number
        try:
            atom_feats["atomic_num"] = atom.GetAtomicNum()
        except Exception:
            atom_feats["atomic_num"] = 0

        # Atomic mass
        try:
            atom_feats["mass"] = atom.GetMass()
        except Exception:
            atom_feats["mass"] = 0.0

        # Valence
        try:
            atom_feats["valence"] = atom.GetTotalValence()
        except Exception:
            atom_feats["valence"] = 0

        # Formal charge
        try:
            atom_feats["formal_charge"] = atom.GetFormalCharge()
        except Exception:
            atom_feats["formal_charge"] = 0

        # Degree
        try:
            atom_feats["degree"] = atom.GetDegree()
        except Exception:
            atom_feats["degree"] = 0

        # Hybridization
        try:
            atom_feats["hybridization"] = str(atom.GetHybridization())
        except Exception:
            atom_feats["hybridization"] = None

        # Aromaticity
        try:
            atom_feats["is_aromatic"] = atom.GetIsAromatic()
        except Exception:
            atom_feats["is_aromatic"] = False

        # Coordinates (if conformer exists)
        if conf:
            try:
                pos = conf.GetAtomPosition(idx)
                atom_feats["coords"] = (pos.x, pos.y, pos.z)
            except Exception:
                atom_feats["coords"] = (0.0, 0.0, 0.0)
        else:
            atom_feats["coords"] = None

        feats.append(atom_feats)

    return feats


feats = safe_molecule_features(mol)  # Example usage
drug_feats = safe_molecule_features(drug)

print("CIF Molecule Features:", feats)
print("Drug Molecule Features:", drug_feats)

Loaded CIF: ./CIF_files/graphene.cif with 8 atoms
Loaded SDF: ./SDF_files/fluorouracil.sdf with 12 atoms
CIF Molecule Features: [{'symbol': 'C', 'atomic_num': 6, 'mass': 12.011, 'valence': 0, 'formal_charge': 0, 'degree': 3, 'hybridization': 'UNSPECIFIED', 'is_aromatic': False, 'coords': (1.6439800002432392e-06, 1.4237284433135415, 0.5000000000000001)}, {'symbol': 'C', 'atomic_num': 6, 'mass': 12.011, 'valence': 0, 'formal_charge': 0, 'degree': 3, 'hybridization': 'UNSPECIFIED', 'is_aromatic': False, 'coords': (-1.2329833560199992, 3.559321108283854, 0.5000000000000003)}, {'symbol': 'C', 'atomic_num': 6, 'mass': 12.011, 'valence': 0, 'formal_charge': 0, 'degree': 3, 'hybridization': 'UNSPECIFIED', 'is_aromatic': False, 'coords': (2.4659716439800006, 1.4237284433135415, 0.5000000000000003)}, {'symbol': 'C', 'atomic_num': 6, 'mass': 12.011, 'valence': 0, 'formal_charge': 0, 'degree': 3, 'hybridization': 'UNSPECIFIED', 'is_aromatic': False, 'coords': (1.2329866439800008, 3.559321108283854

c:\Users\spran\anaconda3\envs\drug\lib\site-packages\pymatgen\core\structure.py:3107: UserWarning: Issues encountered while parsing CIF: 1 fractional coordinates rounded to ideal values to avoid issues with finite precision.
Skipping relative stoichiometry check because CIF does not contain formula keys.
  struct = parser.parse_structures(primitive=primitive)[0]
[11:15:48] 

****
Pre-condition Violation
getValence(ValenceType::EXPLICIT) called without call to calcExplicitValence()
Violation occurred on line 321 in file C:\rdkit\build\temp.win-amd64-cpython-310\Release\rdkit\Code\GraphMol\Atom.cpp
Failed Expression: (which == ValenceType::IMPLICIT || d_explicitValence > -1)
****

[11:15:48] 

****
Pre-condition Violation
getValence(ValenceType::EXPLICIT) called without call to calcExplicitValence()
Violation occurred on line 321 in file C:\rdkit\build\temp.win-amd64-cpython-310\Release\rdkit\Code\GraphMol\Atom.cpp
Failed Expression: (which == ValenceType::IMPLICIT || d_explicitValence >

In [7]:
from torch_geometric.data import Data

class PairedData(Data):
    def __init__(self, data1, data2, y=0.0, weight=1.0):
        super().__init__()
        self.x1 = data1.x
        self.edge_index1 = data1.edge_index
        self.edge_attr1 = data1.edge_attr
        
        self.x2 = data2.x
        self.edge_index2 = data2.edge_index
        self.edge_attr2 = data2.edge_attr
        
        self.y = y  # Target value for the pair
        self.w = torch.tensor([weight], dtype=torch.float)

    def __inc__(self, key, value, *args, **kwargs):
        """Ensures proper indexing when batching."""
        if key == "edge_index1":
            return self.x1.shape[0] if self.x1 is not None else 0
        if key == "edge_index2":
            return self.x2.shape[0] if self.x2 is not None else 0
        return super().__inc__(key, value, *args, **kwargs)
    


In [20]:
import pandas as pd
import os

df = pd.read_csv(r"trial_dataCopy.csv") #TODO

file_path_cif = r"CIF_files" 
file_path_sdf = r"SDF_files" 

materials = []
drugs = []

for cif in df["material"]:
    file = os.path.join(file_path_cif, cif)
    print(f"Processing CIF: {file}")  # Progress tracking
    mol = load_molecule(file)
    materials.append(graph_from_molecule(mol))
print("Finished processing all CIF files.\n")

for sdf in df["drug"]:
    file = os.path.join(file_path_sdf, sdf + ".sdf")
    print(f"Processing SDF: {file}")  # Progress tracking
    supplier = Chem.SDMolSupplier(file)
    mol = supplier[0]
    if mol is None:
        print(f"Warning: Failed to read molecule from {file}")  # Handle errors
    drugs.append(graph_from_molecule(mol))
print("Finished processing all SDF files.\n")

from torch_geometric.utils import add_self_loops

def add_loops_to_data(data):
    edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.x.size(0))
    data.edge_index = edge_index
    return data

print("Pairing materials and drugs...")
paired_data_list = []
for i, (mat, drug, target, weight) in enumerate(zip(materials, drugs, df["y"], df["weight"])):
    print(f"Pairing {i+1}/{len(df)}: Material-{i}, Drug-{i}, Target-{target}")
    print(f"Material graph: {mat}, Drug graph: {drug}")  # Debugging output

    # mat = add_loops_to_data(mat)
    # drug = add_loops_to_data(drug)

    paired_data_list.append(PairedData(mat, drug, target, weight))
    #paired_data_list.append(PairedData(drug, mat, target))


print("Finished pairing all data.")


Processing CIF: CIF_files\antimonene
Loaded CIF: CIF_files\antimonene.cif with 8 atoms
Processing CIF: CIF_files\antimonene
Loaded CIF: CIF_files\antimonene.cif with 8 atoms
Processing CIF: CIF_files\antimonene
Loaded CIF: CIF_files\antimonene.cif with 8 atoms
Processing CIF: CIF_files\BC3
Loaded CIF: CIF_files\BC3.cif with 8 atoms
Processing CIF: CIF_files\BC3
Loaded CIF: CIF_files\BC3.cif with 8 atoms
Processing CIF: CIF_files\Biphenylene
Loaded CIF: CIF_files\Biphenylene.cif with 24 atoms
Processing CIF: CIF_files\BN
Loaded CIF: CIF_files\BN.cif with 2 atoms
Processing CIF: CIF_files\BN
Loaded CIF: CIF_files\BN.cif with 2 atoms
Processing CIF: CIF_files\BN
Loaded CIF: CIF_files\BN.cif with 2 atoms
Processing CIF: CIF_files\BN
Loaded CIF: CIF_files\BN.cif with 2 atoms
Processing CIF: CIF_files\BN
Loaded CIF: CIF_files\BN.cif with 2 atoms
Processing CIF: CIF_files\BN
Loaded CIF: CIF_files\BN.cif with 2 atoms
Processing CIF: CIF_files\BN
Loaded CIF: CIF_files\BN.cif with 2 atoms
Proces

In [10]:
import torch.nn as nn
import torch.optim as optim
from torch_geometric.loader import DataLoader  # Assuming PyG DataLoader

# Define training function
def train(model, train_loader, optimizer, criterion, device):
    model.train()  # Set model to training mode
    total_loss = 0

    for batch in train_loader:

        batch = batch.to(device)  # Move batch to GPU if available
        
        optimizer.zero_grad()  # Reset gradients
        output = model(batch)  # Forward pass

        tensor_x = torch.tensor([batch.y], dtype=torch.float).to(device)  # Convert target to tensor 

        loss = torch.sqrt(criterion(output, tensor_x))  # RMSE Loss
        weighted_loss = (loss * batch.w)
        weighted_loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()
    
    return total_loss / len(train_loader)  # Return average loss

# Define evaluation function
def evaluate(model, val_loader, criterion, device):

    model.eval()  # Set model to evaluation mode
    total_loss = 0

    with torch.no_grad():  # Disable gradient tracking
        for batch in val_loader:
            batch = batch.to(device)
        
            output = model(batch)

            tensor_x = torch.tensor([batch.y], dtype=torch.float).to(device)  # Convert target to tensor 

            loss = torch.sqrt(criterion(output,tensor_x)) # RMSE Loss
            total_loss += np.absolute(loss.item())

    return total_loss / len(val_loader)
    


In [12]:
# === GCN encoder (ignores edge features) ===
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class GCNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
        self.convs.append(GCNConv(hidden_channels, out_channels))
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)  # <--- edge_attr not used

            # print(x)

            if i != len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        # return graph-level embedding by mean-pooling over nodes
        return torch.cat([gap(x,batch=None), gmp(x,batch=None)], dim=1)   # single-graph pooling


# === Regressor that combines two graphs ===
class GCNRegressor(nn.Module):
    def __init__(self, node_dim, hidden_dim=64, gnn_out_dim=128, mlp_hidden=64):
        super().__init__()
        self.encoder1 = GCNEncoder(node_dim, hidden_dim, gnn_out_dim)
        self.encoder2 = GCNEncoder(node_dim, hidden_dim, gnn_out_dim)
        
        self.mlp = nn.Sequential(
            nn.Linear(4 * gnn_out_dim, mlp_hidden),
            nn.ReLU(),
            nn.Linear(mlp_hidden, 1)   # regression output
        )

    def forward(self, data):
        h1 = self.encoder1(data.x1, data.edge_index1)
        h2 = self.encoder2(data.x2, data.edge_index2)
        h = torch.cat([h1, h2], dim=-1)
        out = self.mlp(h)

        # print("Model output before view:", h)  # Debugging output

        return out.view(-1)


In [13]:
from torch_geometric.nn import GATConv

class GATEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4, num_layers=2, dropout=0.5):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout))
        
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_channels * heads, hidden_channels, heads=heads, dropout=dropout))
        
        self.convs.append(GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=dropout))
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i != len(self.convs) - 1:
                x = F.elu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return x

class GATRegressor(nn.Module):
    def __init__(self, node_dim, hidden_dim=64, gnn_out_dim=64):
        super().__init__()
        self.encoder1 = GATEncoder(node_dim, hidden_dim, gnn_out_dim)
        self.encoder2 = GATEncoder(node_dim, hidden_dim, gnn_out_dim)
        self.fc = nn.Sequential(
            nn.Linear(4 * gnn_out_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, data):
        # First graph (material)
        x1 = self.encoder1(data.x1, data.edge_index1)
        h1 = torch.cat([gmp(x1,batch=None), gap(x1, batch=None)], dim=1)


        # Second graph (drug)
        x2 = self.encoder2(data.x2, data.edge_index2)
        h2 = torch.cat([gmp(x2,batch=None), gap(x2, batch=None)], dim=1)


        # Concatenate + predict
        h = torch.cat([h1, h2], dim=-1)
        return self.fc(h).squeeze()


In [14]:
# === Transformer over concatenated graphs (pairwise regression) ===
import torch
import torch.nn as nn
import torch.nn.functional as F

def build_allowed_mask(n1: int, n2: int, edge_index1: torch.Tensor, edge_index2: torch.Tensor, device=None):
    """
    Returns an [L, L] mask of allowed attentions (1=allow, 0=block),
    with adjacency (incl. self-loops) on the diagonal blocks, and all-ones off-diagonal.
    """
    L = n1 + n2
    allowed = torch.zeros((L, L), dtype=torch.bool, device=device)

    # --- material block A1 ---
    A1 = torch.zeros((n1, n1), dtype=torch.bool, device=device)
    if edge_index1.numel() > 0:
        src, dst = edge_index1  # edges j->i
        A1[dst, src] = True
        A1[src, dst] = True   # symmetrize; safe for undirected use
    A1.fill_diagonal_(True)   # self-loops
    allowed[:n1, :n1] = A1

    # --- drug block A2 ---
    A2 = torch.zeros((n2, n2), dtype=torch.bool, device=device)
    if edge_index2.numel() > 0:
        src2, dst2 = edge_index2
        A2[dst2, src2] = True
        A2[src2, dst2] = True
    A2.fill_diagonal_(True)
    allowed[n1:, n1:] = A2

    # --- off-diagonals: full cross attention ---
    allowed[:n1, n1:] = True
    allowed[n1:, :n1] = True

    return allowed  # [L, L] bool

class TransformerRegressor(nn.Module):
    """
    - Projects node features to d_model
    - Adds token-type embedding (0=material, 1=drug)
    - Runs Transformer encoder with a custom attention mask
    - Pools per graph and regresses a scalar
    """
    def __init__(self, node_dim: int, d_model: int = 128, nhead: int = 8,
                 num_layers: int = 4, dim_feedforward: int = 256, dropout: float = 0.1):
        super().__init__()
        self.in_proj = nn.Linear(node_dim, d_model)
        self.type_embed = nn.Embedding(2, d_model)  # 0: material, 1: drug

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.Linear(4 * d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, 1)
        )

    def forward(self, data):
        """
        data: PairedData with fields x1, edge_index1, x2, edge_index2, y (float)
        """
        device = next(self.parameters()).device
        x1 = data.x1.to(device).float()
        x2 = data.x2.to(device).float()
        ei1 = data.edge_index1.to(device).long() if data.edge_index1 is not None else torch.empty((2,0), dtype=torch.long, device=device)
        ei2 = data.edge_index2.to(device).long() if data.edge_index2 is not None else torch.empty((2,0), dtype=torch.long, device=device)

        n1, n2 = x1.size(0), x2.size(0)
        L = n1 + n2

        # Node embeddings + token type embeddings
        h1 = self.in_proj(x1) + self.type_embed(torch.zeros(n1, dtype=torch.long, device=device))   # material
        h2 = self.in_proj(x2) + self.type_embed(torch.ones(n2, dtype=torch.long, device=device))    # drug
        H = torch.cat([h1, h2], dim=0).unsqueeze(0)  # [1, L, d_model] batch_first

        # Build allowed-attention mask and convert to additive attn_mask
        allowed = build_allowed_mask(n1, n2, ei1, ei2, device=device)  # [L, L] bool
        # PyTorch expects attn_mask where True=mask OR a float with -inf where masked.
        # We'll use float additive mask: 0 for allowed, -inf for blocked.
        attn_mask = (~allowed).float() * -1e9  # [L, L]

        # Encoder
        Z = self.encoder(H, mask=attn_mask)  # [1, L, d_model]
        Z_nodes = Z[0]  # [L, d_model]

        z1_mean = Z_nodes[:n1].mean(dim=0, keepdim=True)
        z1_max  = Z_nodes[:n1].max(dim=0, keepdim=True).values
        z1 = torch.cat([z1_mean, z1_max], dim=-1)  # [1, 2*d_model]

        z2_mean = Z_nodes[n1:].mean(dim=0, keepdim=True)
        z2_max  = Z_nodes[n1:].max(dim=0, keepdim=True).values
        z2 = torch.cat([z2_mean, z2_max], dim=-1)  # [1, 2*d_model]

        h = torch.cat([z1, z2], dim=-1)  # [1, 4*d_model]

        out = self.head(h)

        return out




In [ ]:
from sklearn.model_selection import KFold
from torch.utils.data import Subset

k = 7
num_epochs = 100
kfold = KFold(n_splits=k, shuffle=True, random_state=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
val_losses = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(paired_data_list)):
    print(f"\nFold {fold+1}/{k}")

    # Subset works fine because paired_data_list is indexable
    train_loader = Subset(paired_data_list, train_idx)
    val_loader = Subset(paired_data_list, val_idx)

    print(f"Training set size: {len(train_loader)}")
    print(f"Validation set size: {len(val_loader)}")

    # Fresh GCN model for each fold
    # model = GCNRegressor(node_dim=node_dim, hidden_dim=64, gnn_out_dim=128).to(device)
    # model = GATRegressor(node_dim=node_dim, hidden_dim=64, gnn_out_dim=128).to(device)
    model = TransformerRegressor(node_dim=node_dim, d_model=128, nhead=8, num_layers=4, dim_feedforward=256).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
    criterion = torch.nn.MSELoss()

    for epoch in range(0, num_epochs + 1):
        train_loss = train(model, train_loader, optimizer, criterion, device)

        if epoch % 10 == 0:
            val_loss = evaluate(model, val_loader, criterion, device)
            print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

    val_loss = evaluate(model, val_loader, criterion, device)
    val_losses.append(val_loss)

# Final result
mean_val_loss = sum(val_losses) / len(val_losses)
print(f"\n✅ K-Fold Cross-Validation Complete! Mean Validation Loss = {mean_val_loss:.4f}")


Fold 1/7
Training set size: 42
Validation set size: 7
Epoch 0: Train Loss = 2.4542, Val Loss = 0.8124
Epoch 10: Train Loss = 1.0265, Val Loss = 0.4458
Epoch 20: Train Loss = 0.9571, Val Loss = 0.6052
Epoch 30: Train Loss = 0.9404, Val Loss = 0.5133
Epoch 40: Train Loss = 0.9397, Val Loss = 0.5459
Epoch 50: Train Loss = 0.9513, Val Loss = 0.5256
Epoch 60: Train Loss = 0.9376, Val Loss = 0.5411
Epoch 70: Train Loss = 0.9135, Val Loss = 0.4746
Epoch 80: Train Loss = 0.9724, Val Loss = 0.4824
Epoch 90: Train Loss = 0.9233, Val Loss = 0.4602
Epoch 100: Train Loss = 0.9180, Val Loss = 0.4138

Fold 2/7
Training set size: 42
Validation set size: 7
Epoch 0: Train Loss = 1.7945, Val Loss = 1.3774
Epoch 10: Train Loss = 0.8764, Val Loss = 1.2829
Epoch 20: Train Loss = 0.8649, Val Loss = 1.2274
Epoch 30: Train Loss = 0.8425, Val Loss = 1.2539
Epoch 40: Train Loss = 0.8289, Val Loss = 1.1812
Epoch 50: Train Loss = 0.8410, Val Loss = 1.1718
Epoch 60: Train Loss = 0.8171, Val Loss = 1.2794
Epoch 70:

In [21]:
num_epochs = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
complete_data_loader = paired_data_list

# Fresh Models for final training

# model = GCNRegressor(node_dim=node_dim, hidden_dim=64, gnn_out_dim=128).to(device)
# model = GATRegressor(node_dim=node_dim, hidden_dim=64, gnn_out_dim=128).to(device)
# model = MPNNRegressor(node_dim=node_dim, edge_dim=edge_dim, hidden_dim=64, gnn_out_dim=128).to(device)
model = TransformerRegressor(node_dim=node_dim, d_model=128, nhead=8, num_layers=4, dim_feedforward=256).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-5)
criterion = torch.nn.MSELoss()
for epoch in range(0, num_epochs + 1):
    train_loss = train(model, complete_data_loader, optimizer, criterion, device)

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}")


Epoch 0: Train Loss = 1.1432
Epoch 10: Train Loss = 0.9784
Epoch 20: Train Loss = 0.9339
Epoch 30: Train Loss = 1.0160
Epoch 40: Train Loss = 0.9084
Epoch 50: Train Loss = 0.9421
Epoch 60: Train Loss = 0.8587
Epoch 70: Train Loss = 0.8970
Epoch 80: Train Loss = 0.8438
Epoch 90: Train Loss = 0.8224
Epoch 100: Train Loss = 0.7837


In [18]:
import os
import pandas as pd
from rdkit import Chem

# ---- Load Test Data ----
df_test = pd.read_csv(r"trial_dataCopy.csv")

file_path_cif = r"CIF_files"
file_path_sdf = r"SDF_files"

materials_test = []
drugs_test = []

# ---- Process CIFs ----
for cif in df_test["material"]:
    file = os.path.join(file_path_cif, cif + ".cif")
    # print(f"Processing CIF: {file}")
    mol = load_molecule(file)
    materials_test.append(graph_from_molecule(mol))
print("Finished processing all test CIF files.\n")

# ---- Process SDFs ----
for sdf in df_test["drug"]:
    file = os.path.join(file_path_sdf, sdf + ".sdf")
    # print(f"Processing SDF: {file}")
    supplier = Chem.SDMolSupplier(file)
    mol = supplier[0]
    if mol is None:
        print(f"Warning: Failed to read molecule from {file}")
    drugs_test.append(graph_from_molecule(mol))
print("Finished processing all test SDF files.\n")

# ---- Pair them ----
paired_data_list_test = []
for i, (mat, drug) in enumerate(zip(materials_test, drugs_test)):
    # print(f"Pairing {i+1}/{len(df_test)}: Material-{i}, Drug-{i}")
    paired_data_list_test.append(PairedData(mat, drug, None))  # no target/weight

print("Finished pairing all test data.\n")

# print(paired_data_list_test[5].x2)
# random_model = GCNRegressor(node_dim=node_dim, hidden_dim=64, gnn_out_dim=128).to(device)

# ---- Run Predictions ----
model.eval()
with torch.no_grad():
    for i, pair in enumerate(paired_data_list_test):
        pair = pair.to(device)
        pred = model(pair)   # Assuming your model takes PairedData

        # print(pair.x1)

        print(f"Prediction for pair {i+1}: {pred.item():.4f}")



⚠️ No CIF or SDF found for CIF_files\antimonene.cif


AttributeError: 'NoneType' object has no attribute 'GetAtoms'

In [23]:
import os
import pandas as pd
from rdkit import Chem
from torch_geometric.loader import DataLoader

# --- assumes you already have: ---
# 1. a function `mol_to_graph(mol)` that converts an RDKit mol -> PyG Data
# 2. your trained model `model`
# 3. your material graph `material` (a PyG Data object)

def process_sdf_and_predict(sdf_file, material, model, csv_out="predictions.csv"):
    # --- Load molecules from SDF ---
    suppl = Chem.SDMolSupplier(sdf_file, removeHs=False)
    drugs = [mol for mol in suppl if mol is not None]

    # --- Build material graph once ---
    material_graph = graph_from_molecule(material)
    mat_graph = add_loops_to_data(material_graph)
    print("Material graph:", mat_graph)

    paired_data_list = []
    meta = []  # (name, num_atoms)

    # --- Convert each drug ---
    for i, drug in enumerate(drugs):
        try:
            # --- name ---
            if drug.HasProp("Name"):
                name = drug.GetProp("Name")
            elif drug.HasProp("_Name"):
                name = drug.GetProp("_Name")
            else:
                name = f"drug_{i}"

            num_atoms = drug.GetNumAtoms()

            # --- convert to graph ---
            drug_graph = graph_from_molecule(drug)
            drug_graph = add_loops_to_data(drug_graph)

            # skip if graph looks broken
            if drug_graph.x is None or drug_graph.x.size(0) == 0 or drug_graph.x.size(1) == 0:
                print(f"Skipping {name}: invalid drug graph")
                continue

            # --- pair with material ---
            paired = PairedData(mat_graph, drug_graph, y=0.0, weight=1.0)
            print("Prepaired pair:", paired)

            paired_data_list.append(paired)
            meta.append((name, num_atoms))

        except Exception as e:
            print(f"Skipping drug {i} due to error: {e}")
            continue

    # --- Predict ---
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    preds = []
    names = []
    atom_counts = []

    with torch.no_grad():
        for (batch, (name, num_atoms)) in zip(paired_data_list, meta):
            try:
                batch = batch.to(device)
                out = model(batch)
                pred = out.view(-1).item()

                preds.append(pred)
                names.append(name)
                atom_counts.append(num_atoms)

            except Exception as e:
                print(f"Skipping {name} due to model error: {e}")
                continue
                


    # --- Save to CSV ---
    df = pd.DataFrame({
        "name": names,
        "num_atoms": atom_counts,
        "pred_energy": preds
    })
    df.to_csv(csv_out, index=False)
    print(f"Saved {len(df)} predictions to {csv_out}")

# --- usage ---
mat = cif_to_mol("CIF_files/Graphyne.cif")
# random_model = GCNRegressor(node_dim=node_dim, hidden_dim=64, gnn_out_dim=128)  # Replace with your trained model
process_sdf_and_predict("Approveddrugslibrary.sdf", mat, model, "drug_predictions.csv")


Material graph: Data(x=[48, 21], edge_index=[2, 432], edge_attr=[384, 7])
Prepaired pair: PairedData(x1=[48, 21], edge_index1=[2, 432], edge_attr1=[384, 7], x2=[15, 21], edge_index2=[2, 43], edge_attr2=[28, 7], y=0.0, w=[1])
Prepaired pair: PairedData(x1=[48, 21], edge_index1=[2, 432], edge_attr1=[384, 7], x2=[15, 21], edge_index2=[2, 47], edge_attr2=[32, 7], y=0.0, w=[1])
Prepaired pair: PairedData(x1=[48, 21], edge_index1=[2, 432], edge_attr1=[384, 7], x2=[35, 21], edge_index2=[2, 113], edge_attr2=[78, 7], y=0.0, w=[1])
Prepaired pair: PairedData(x1=[48, 21], edge_index1=[2, 432], edge_attr1=[384, 7], x2=[34, 21], edge_index2=[2, 110], edge_attr2=[76, 7], y=0.0, w=[1])
Prepaired pair: PairedData(x1=[48, 21], edge_index1=[2, 432], edge_attr1=[384, 7], x2=[32, 21], edge_index2=[2, 100], edge_attr2=[68, 7], y=0.0, w=[1])
Prepaired pair: PairedData(x1=[48, 21], edge_index1=[2, 432], edge_attr1=[384, 7], x2=[25, 21], edge_index2=[2, 75], edge_attr2=[50, 7], y=0.0, w=[1])
Prepaired pair: P